In [ ]:
import SimpleITK as sitk
import numpy as np
import os

In [ ]:
def bounding_box(colon_mask):

   # Convert the colon mask to a numpy array
    colon_array = sitk.GetArrayFromImage(colon_mask)  # Shape: [Z, Y, X]

    # Find the indices where the colon mask is non-zero
    colon_indices = np.nonzero(colon_array)

    # Extract the Z coordinates
    Y_coords = colon_indices[1]
    min_Y = np.min(Y_coords)

    # Get the spacing in the Z-direction (slice thickness)
    voxel_height = colon_mask.GetSpacing()[1]  # In millimeters

    # Define the physical height of the rectum area (e.g., 100 mm)
    rectum_height_mm = 40  # Adjust based on anatomical knowledge

    # Calculate the number of slices to include
    num_voxels = int(rectum_height_mm / voxel_height)

    # Define the upper Z limit for the rectum area
    rectum_max_Y = min_Y + num_voxels
    rectum_max_Y = min(rectum_max_Y, colon_array.shape[1] - 1)  # Ensure within bounds

    # Initialize an array to hold the rectum mask
    rectum_array = np.zeros_like(colon_array)

    # Copy the relevant slices from the colon mask
    rectum_array[:, min_Y:rectum_max_Y+1, :] = colon_array[:, min_Y:rectum_max_Y+1, :]

    # Convert the numpy array back to a SimpleITK image
    rectum_mask = sitk.GetImageFromArray(rectum_array)

    # Copy spatial information from the original colon mask
    rectum_mask.CopyInformation(colon_mask)

    # Find the non-zero indices in the rectum mask
    rectum_indices = np.nonzero(rectum_array)

    # Get the minimum and maximum indices for each dimension
    min_Z_rect, max_Z_rect = np.min(rectum_indices[0]), np.max(rectum_indices[0])
    min_Y_rect, max_Y_rect = np.min(rectum_indices[1]), np.max(rectum_indices[1])
    min_X_rect, max_X_rect = np.min(rectum_indices[2]), np.max(rectum_indices[2])
   
    # Define the expansion size (in voxels)
    expansion = 20  # Adjust based on desired margin

    # Get the dimensions of the image
    image_size = colon_mask.GetSize()  # Returns (X, Y, Z)

    # Expand the bounding box, ensuring it stays within image bounds
    min_Z = max(min_Z_rect - expansion, 0)
    max_Z = min(max_Z_rect + expansion, image_size[2] - 1)

    min_Y = max(min_Y_rect - expansion, 0)
    max_Y = min(max_Y_rect + expansion, image_size[1] - 1)

    min_X = max(min_X_rect - expansion, 0)
    max_X = min(max_X_rect + expansion, image_size[0] - 1)
    
    return min_Z, max_Z, min_Y, max_Y, min_X, max_X

In [ ]:
def extract_patches(minZ, maxZ, minY, maxY, minX, maxX, mr_image):
    # Define the size and index for the Region of Interest (ROI)
    roi_size = [
        int(maxX - minX + 1),
        int(maxY - minY + 1),
        int(maxZ - minZ + 1)
    ]

    roi_index = [
        int(minX),
        int(minY),
        int(minZ)
    ]

    # Set up the RegionOfInterestImageFilter
    roi_filter = sitk.RegionOfInterestImageFilter()
    roi_filter.SetSize(roi_size)
    roi_filter.SetIndex(roi_index)

    # Extract the patch
    rectum_patch = roi_filter.Execute(mr_image)
    return rectum_patch

In [ ]:
path1 = r'/path/to/workspace/classification_model_originalimage/totalsegment_work/resampled/male_test'
path2 = r'/path/to/workspace/classification_model_originalimage/totalsegment_work/resampled/male_test_patches'
save_path = r'/path/to/workspace/classification_model_originalimage/totalsegment_work/resampled/left_test_patches'

for i in os.listdir(path1):
    filename = os.listdir(path2)
    i_new = i + '.nii.gz'
    
    if i_new not in filename:  #find error male cases when doing rectum patches(because of missing prostate mask)
        # print(i_new)
        i_new2 = i_new.replace('.nii.gz', '') #foldername
        file_path = os.path.join(path1, i_new2)
        image_path =  os.path.join(file_path, 'resampled_image.nii.gz')
        colon_path =  os.path.join(file_path, 'colon.nii.gz')

        image = sitk.ReadImage(image_path)
        colon_img = sitk.ReadImage(colon_path)

        try:
            min_Z, max_Z, min_Y, max_Y, min_X, max_X = bounding_box(colon_img)
            rectum_patch = extract_patches(min_Z, max_Z, min_Y, max_Y, min_X, max_X, image)
    
            # Save the extracted patch
            new_foldername = i_new
            save_path_new = os.path.join(save_path, new_foldername)
            sitk.WriteImage(rectum_patch, save_path_new)
        
        except Exception as e:
            print(f"An error occurred: {e}")
            print("patient number:", i_new2)
        
    
print('done')

